# Stage 1B: U-Net untuk Segmentasi Hard Exudate

**Kenapa U-Net, bukan Mask R-CNN?**
- HE = banyak bercak kecil tersebar → semantic segmentation, bukan instance segmentation
- Mask R-CNN hanya mencapai IoU ~0.50 untuk HE (terlalu sulit per-instance)
- Paper dosen (INSERT 2023) juga pakai U-Net untuk HE

**Pipeline:**
1. Load gambar fundus + GT mask HE
2. Resize ke 256x256 (sesuai paper dosen)
3. Online augmentation (Albumentations)
4. Train U-Net (encoder: pre-trained ResNet34)
5. Evaluasi: IoU, Sensitivity, Specificity

**Baseline paper:** Acc=0.993, Sens=0.454, Spec=0.997

## 1. Install & Import

In [2]:
# Install segmentation_models_pytorch jika belum ada
try:
    import segmentation_models_pytorch as smp
except ImportError:
    import subprocess, sys
    # Coba uv dulu (untuk venv tanpa pip), fallback ke pip
    try:
        subprocess.check_call(["uv", "pip", "install", "-q", "segmentation-models-pytorch"])
    except FileNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"])
    import segmentation_models_pytorch as smp

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision

import numpy as np
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from pathlib import Path
import time
import json

print(f"PyTorch  : {torch.__version__}")
print(f"SMP      : {smp.__version__}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device   : {DEVICE}")

PyTorch  : 2.10.0
SMP      : 0.5.0
Device   : mps


## 2. Path Konfigurasi

In [ ]:
BASE_DIR = Path(".")

ORIG_TRAIN = BASE_DIR / "1. Original Images" / "a. Training Set"
ORIG_TEST  = BASE_DIR / "1. Original Images" / "b. Testing Set"

GT_HE_TRAIN = BASE_DIR / "2. All Segmentation Groundtruths" / "a. Training Set" / "3. Hard Exudates"
GT_HE_TEST  = BASE_DIR / "2. All Segmentation Groundtruths" / "b. Testing Set"  / "3. Hard Exudates"

HE_SUFFIX = "_EX.tif"  # IDRiD uses _EX suffix

IMG_SIZE = 512

print("Path konfigurasi OK")
print(f"  Image size   : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Train images : {len(list(ORIG_TRAIN.glob('IDRiD_*.jpg')))}")
print(f"  Test images  : {len(list(ORIG_TEST.glob('IDRiD_*.jpg')))}")
print(f"  Train HE GT  : {len(list(GT_HE_TRAIN.glob('*.tif')))}")
print(f"  Test HE GT   : {len(list(GT_HE_TEST.glob('*.tif')))}")

## 3. Dataset Class

In [4]:
class IDRiDHEDataset(Dataset):
    """
    Dataset untuk semantic segmentation Hard Exudate.
    Output: image tensor [3, H, W], mask tensor [1, H, W] (binary 0/1)
    """

    def __init__(self, img_dir, gt_dir, gt_suffix, transform=None, img_size=512):
        self.img_dir = Path(img_dir)
        self.gt_dir = Path(gt_dir)
        self.gt_suffix = gt_suffix
        self.transform = transform
        self.img_size = img_size

        self.samples = []
        for img_path in sorted(self.img_dir.glob("IDRiD_*.jpg")):
            stem = img_path.stem
            gt_path = self.gt_dir / f"{stem}{self.gt_suffix}"
            self.samples.append({
                "img_path": img_path,
                "gt_path": gt_path if gt_path.exists() else None,
                "stem": stem,
            })

        n_with_gt = sum(1 for s in self.samples if s["gt_path"] is not None)
        print(f"  Dataset: {len(self.samples)} gambar, {n_with_gt} dengan GT")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Load image
        image = cv2.imread(str(sample["img_path"]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))

        # Load mask
        if sample["gt_path"] is not None:
            mask = cv2.imread(str(sample["gt_path"]), cv2.IMREAD_UNCHANGED)
            if mask.ndim == 3:
                mask = mask[:, :, :3].max(axis=2)
            mask = (mask > 10).astype(np.float32)
            mask = cv2.resize(mask, (self.img_size, self.img_size),
                              interpolation=cv2.INTER_NEAREST)
        else:
            mask = np.zeros((self.img_size, self.img_size), dtype=np.float32)

        # Augmentation
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        else:
            # Default: normalize & to tensor
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mask = torch.from_numpy(mask)

        # Mask shape: [1, H, W]
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        return image, mask

print("Dataset class OK")

Dataset class OK


## 4. Augmentasi

In [5]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.1,
        scale_limit=0.2,
        rotate_limit=45,
        border_mode=cv2.BORDER_CONSTANT,
        p=0.5,
    ),
    A.ElasticTransform(alpha=120, sigma=12, p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(var_limit=(10, 50), p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

test_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print("Augmentasi OK")

Augmentasi OK


/Users/mac/Documents/Kuliah/Semester 6/Skripsi/.venv/lib/python3.14/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/var/folders/p3/0mc5zsd91bdcw4p37s0rxmf80000gn/T/ipykernel_7342/759778290.py:14: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 50), p=0.2),


## 5. Model: U-Net + ResNet34 Encoder

In [6]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,  # binary: HE or not
    activation=None,  # raw logits, sigmoid diterapkan di loss/inference
)

model = model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Total params     : 24,436,369
Trainable params : 24,436,369


## 6. Loss Function: BCE + Dice

In [ ]:
class BCEDiceLoss(nn.Module):
    """
    Kombinasi BCE + Dice Loss.
    - pos_weight: beri bobot lebih ke piksel positif (HE) karena sangat sedikit
    - Dice menangani class imbalance secara natural
    """

    def __init__(self, bce_weight=0.3, dice_weight=0.7, smooth=1.0, pos_weight=10.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth
        # pos_weight: HE pixels sangat sedikit (~1% image), beri bobot 10x
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))

    def forward(self, logits, targets):
        # BCE Loss (dengan pos_weight)
        bce_loss = self.bce(logits.to(targets.device), targets)

        # Dice Loss
        probs = torch.sigmoid(logits)
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)

        intersection = (probs_flat * targets_flat).sum()
        dice = (2.0 * intersection + self.smooth) / (
            probs_flat.sum() + targets_flat.sum() + self.smooth
        )
        dice_loss = 1.0 - dice

        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# pos_weight=10: HE area ~1% of image, jadi positif dihargai 10x lipat
criterion = BCEDiceLoss(bce_weight=0.3, dice_weight=0.7, pos_weight=10.0)
print("Loss function OK (BCE weight=0.3, Dice weight=0.7, pos_weight=10)")

## 7. Metrik Evaluasi

In [8]:
def compute_metrics(pred_mask, gt_mask):
    """
    Compute IoU, Sensitivity (Recall), Specificity.
    Input: binary masks (0/1), flattened or 2D.
    """
    pred = pred_mask.flatten().astype(bool)
    gt = gt_mask.flatten().astype(bool)

    tp = (pred & gt).sum()
    fp = (pred & ~gt).sum()
    fn = (~pred & gt).sum()
    tn = (~pred & ~gt).sum()

    # IoU
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 1.0

    # Sensitivity (Recall) = TP / (TP + FN)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 1.0

    # Specificity = TN / (TN + FP)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 1.0

    # Accuracy
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0

    return {
        "iou": float(iou),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "accuracy": float(accuracy),
    }

print("Metrik OK")

Metrik OK


## 8. Training & Evaluation Functions

In [9]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    n_batches = 0

    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, dataloader, device, threshold=0.5):
    model.eval()
    all_metrics = []

    for images, masks in dataloader:
        images = images.to(device)
        logits = model(images)
        preds = (torch.sigmoid(logits) > threshold).cpu().numpy().astype(np.uint8)
        gt = masks.cpu().numpy().astype(np.uint8)

        for i in range(preds.shape[0]):
            m = compute_metrics(preds[i], gt[i])
            all_metrics.append(m)

    # Average metrics
    avg = {}
    for key in all_metrics[0]:
        avg[key] = np.mean([m[key] for m in all_metrics])

    return avg

print("Training & evaluation functions OK")

Training & evaluation functions OK


## 9. Setup DataLoaders

In [10]:
print("[Train set]")
train_dataset = IDRiDHEDataset(
    img_dir=ORIG_TRAIN,
    gt_dir=GT_HE_TRAIN,
    gt_suffix=HE_SUFFIX,
    transform=train_transform,
    img_size=IMG_SIZE,
)

print("[Test set]")
test_dataset = IDRiDHEDataset(
    img_dir=ORIG_TEST,
    gt_dir=GT_HE_TEST,
    gt_suffix=HE_SUFFIX,
    transform=test_transform,
    img_size=IMG_SIZE,
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False, num_workers=0)

# Sanity check
img, msk = train_dataset[0]
print(f"\nSanity check:")
print(f"  Image shape: {img.shape}, dtype: {img.dtype}")
print(f"  Mask shape : {msk.shape}, dtype: {msk.dtype}")
print(f"  Mask range : [{msk.min():.1f}, {msk.max():.1f}]")

[Train set]
  Dataset: 54 gambar, 45 dengan GT
[Test set]
  Dataset: 27 gambar, 27 dengan GT

Sanity check:
  Image shape: torch.Size([3, 512, 512]), dtype: torch.float32
  Mask shape : torch.Size([1, 512, 512]), dtype: torch.float32
  Mask range : [0.0, 0.0]


## 10. Training

In [ ]:
# === Model: UNet++ dengan EfficientNet-B3 encoder ===
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None,
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model        : UNet++ + EfficientNet-B3")
print(f"Total params : {total_params:,}")

# === Hyperparameters ===
NUM_EPOCHS = 150
LR = 1e-4
PATIENCE = 25

# === Optimizer & Scheduler ===
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2, eta_min=1e-6
)

# === Training loop ===
save_dir = Path("runs/unetpp_he")
save_dir.mkdir(parents=True, exist_ok=True)

best_iou = 0.0
best_epoch = 0
history = {"train_loss": [], "test_iou": [], "test_sensitivity": [],
           "test_specificity": [], "lr": []}

print(f"\n{'='*65}")
print(f"  TRAINING UNet++ untuk Hard Exudate (v3)")
print(f"{'='*65}")
print(f"  Encoder    : EfficientNet-B3 (ImageNet)")
print(f"  Epochs     : {NUM_EPOCHS}")
print(f"  Batch size : 4")
print(f"  LR         : {LR}")
print(f"  Scheduler  : CosineAnnealingWarmRestarts (T0=20)")
print(f"  Image size : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Patience   : {PATIENCE}")
print(f"  Loss       : BCE(w=0.3, pos_weight=10) + Dice(w=0.7)")
print(f"  Device     : {DEVICE}")
print()

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()

    # Train
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)

    # Evaluate
    metrics = evaluate(model, test_loader, DEVICE)

    # Scheduler step
    scheduler.step(epoch)
    current_lr = optimizer.param_groups[0]["lr"]

    # History
    history["train_loss"].append(train_loss)
    history["test_iou"].append(metrics["iou"])
    history["test_sensitivity"].append(metrics["sensitivity"])
    history["test_specificity"].append(metrics["specificity"])
    history["lr"].append(current_lr)

    elapsed = time.time() - start

    # Save best
    marker = ""
    if metrics["iou"] > best_iou:
        best_iou = metrics["iou"]
        best_epoch = epoch
        torch.save(model.state_dict(), save_dir / "best.pt")
        marker = " ★ BEST"

    print(
        f"  Epoch {epoch:3d}/{NUM_EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"IoU: {metrics['iou']:.4f} | "
        f"Sens: {metrics['sensitivity']:.4f} | "
        f"Spec: {metrics['specificity']:.4f} | "
        f"LR: {current_lr:.1e} | "
        f"{elapsed:.1f}s{marker}"
    )

    # Early stopping
    if epoch - best_epoch >= PATIENCE:
        print(f"\n  Early stopping at epoch {epoch} (best was epoch {best_epoch})")
        break

# Save last & history
torch.save(model.state_dict(), save_dir / "last.pt")
with open(save_dir / "history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"\n  Best IoU: {best_iou:.4f} (epoch {best_epoch})")
print(f"  Model saved to {save_dir}")

## 11. Visualisasi Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history["train_loss"], "b-")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

# IoU
axes[1].plot(history["test_iou"], "r-")
axes[1].set_title(f"Test IoU (best: {best_iou:.4f})")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("IoU")
axes[1].grid(True, alpha=0.3)

# Sensitivity & Specificity
axes[2].plot(history["test_sensitivity"], "g-", label="Sensitivity")
axes[2].plot(history["test_specificity"], "m-", label="Specificity")
axes[2].set_title("Sensitivity & Specificity")
axes[2].set_xlabel("Epoch")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("visualisasi_training_unet_he.png", dpi=100, bbox_inches="tight")
plt.show()
print("Visualisasi disimpan")

## 12. Evaluasi Detail per Gambar

In [ ]:
# Load best model
best_model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None,
)
best_model.load_state_dict(torch.load("runs/unetpp_he/best.pt", map_location=DEVICE, weights_only=True))
best_model = best_model.to(DEVICE)
best_model.eval()

print(f"{'='*60}")
print(f"  EVALUASI DETAIL: Hard Exudate (UNet++)")
print(f"{'='*60}")

all_metrics = []
with torch.no_grad():
    for idx in range(len(test_dataset)):
        img, gt_mask = test_dataset[idx]
        stem = test_dataset.samples[idx]["stem"]

        logits = best_model(img.unsqueeze(0).to(DEVICE))
        pred = (torch.sigmoid(logits) > 0.5).squeeze().cpu().numpy().astype(np.uint8)
        gt = gt_mask.squeeze().numpy().astype(np.uint8)

        m = compute_metrics(pred, gt)
        all_metrics.append(m)
        print(f"  {stem}: IoU={m['iou']:.4f} | Sens={m['sensitivity']:.4f} | Spec={m['specificity']:.4f}")

# Average
avg_iou  = np.mean([m["iou"] for m in all_metrics])
avg_sens = np.mean([m["sensitivity"] for m in all_metrics])
avg_spec = np.mean([m["specificity"] for m in all_metrics])
avg_acc  = np.mean([m["accuracy"] for m in all_metrics])

print(f"\n  Average IoU         : {avg_iou:.4f}")
print(f"  Average Sensitivity : {avg_sens:.4f}")
print(f"  Average Specificity : {avg_spec:.4f}")
print(f"  Average Accuracy    : {avg_acc:.4f}")
print(f"\n  Baseline paper (U-Net):")
print(f"    Accuracy=0.993, Sensitivity=0.454, Specificity=0.997")

## 13. Visualisasi Prediksi vs Ground Truth

In [ ]:
# Ambil 6 gambar yang punya GT
sample_indices = [
    i for i, s in enumerate(test_dataset.samples)
    if s["gt_path"] is not None
][:6]

# Denormalize helper
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(len(sample_indices), 4, figsize=(20, 5 * len(sample_indices)))

with torch.no_grad():
    for row, idx in enumerate(sample_indices):
        img_tensor, gt_mask = test_dataset[idx]
        stem = test_dataset.samples[idx]["stem"]

        # Predict
        logits = best_model(img_tensor.unsqueeze(0).to(DEVICE))
        pred_mask = (torch.sigmoid(logits) > 0.5).squeeze().cpu().numpy().astype(np.uint8)
        gt_np = gt_mask.squeeze().numpy().astype(np.uint8)

        # Denormalize image
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = (img_np * std + mean).clip(0, 1)

        # IoU
        m = compute_metrics(pred_mask, gt_np)

        # Col 0: Original
        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title(f"{stem} — Original")
        axes[row, 0].axis("off")

        # Col 1: GT mask
        gt_overlay = img_np.copy()
        gt_overlay[gt_np > 0] = [0, 1, 0]
        axes[row, 1].imshow(gt_overlay)
        axes[row, 1].set_title(f"{stem} — Ground Truth")
        axes[row, 1].axis("off")

        # Col 2: Prediction overlay
        pred_overlay = img_np.copy()
        pred_overlay[pred_mask > 0] = [1, 0, 0]
        axes[row, 2].imshow(pred_overlay)
        axes[row, 2].set_title(f"{stem} — Prediction (IoU: {m['iou']:.3f})")
        axes[row, 2].axis("off")

        # Col 3: Side-by-side masks
        comparison = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        comparison[gt_np > 0, 1] = 1.0       # GT = green
        comparison[pred_mask > 0, 0] = 1.0    # Pred = red
        overlap = (gt_np > 0) & (pred_mask > 0)
        comparison[overlap] = [1.0, 1.0, 0.0]  # Overlap = yellow
        axes[row, 3].imshow(comparison)
        axes[row, 3].set_title(f"Green=GT, Red=Pred, Yellow=Overlap")
        axes[row, 3].axis("off")

plt.suptitle("U-Net — Hard Exudate Segmentation", fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig("visualisasi_prediksi_unet_he.png", dpi=100, bbox_inches="tight")
plt.show()
print("Visualisasi disimpan ke visualisasi_prediksi_unet_he.png")

## 14. Ringkasan

In [ ]:
print("="*60)
print("  RINGKASAN STAGE 1B: U-Net Hard Exudate")
print("="*60)
print()
print(f"  {'Metrik':<15} {'Kami':>10} {'Baseline':>10}")
print(f"  {'-'*15} {'-'*10} {'-'*10}")
print(f"  {'IoU':<15} {avg_iou:>10.4f} {'N/A':>10}")
print(f"  {'Accuracy':<15} {avg_acc:>10.4f} {'0.993':>10}")
print(f"  {'Sensitivity':<15} {avg_sens:>10.4f} {'0.454':>10}")
print(f"  {'Specificity':<15} {avg_spec:>10.4f} {'0.997':>10}")
print()
print("  Model: runs/unet_he/best.pt")
print()
print("  Stage 1 Complete:")
print("    - OD: Mask R-CNN → runs/mask_rcnn_od/best.pt")
print("    - HE: U-Net      → runs/unet_he/best.pt")
print("  → Lanjut ke Stage 2: Preprocessing (Blackout + CLAHE)")